# Route E: redox across the mitotic-to-endocycle switch in the fly ovary

Follicle cells divide until stage 6, then endocycle. Both are the same cell type in the same tissue, so this is a cleaner test than the salivary gland. Pick the dataset in Cell 1, then **Runtime → Run all**. Data download from GEO into *Colab Notebooks/route_e_ovary*.

In [ ]:
#@title Cell 1 - Settings and connect Google Drive
import gzip, io, os, re, subprocess, tarfile, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print

#@markdown **GSE146040**: adult ovary single cells, Jevitt et al. 2020 (Deng lab), ~35,000 cells.
#@markdown **GSE136162**: adult ovary single cells, Rust et al. 2020 (Nystul lab).
#@markdown Both hold mitotic follicle cells (stages 1-6) and endocycling follicle cells (stages 7-10) from the same tissue.
GSE = "GSE146040"  #@param ["GSE146040", "GSE136162"]
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_e_ovary"  #@param {type:"string"}
GTF_PATH = "/content/drive/MyDrive/Colab Notebooks/route_b_fly/Drosophila_melanogaster.gtf.gz"  #@param {type:"string"}
N_CLUSTERS = 18  #@param {type:"integer"}
MIN_GENES = 400  #@param {type:"integer"}
MIN_COUNTS = 1000  #@param {type:"integer"}
N_RANDOM_SETS = 2000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}
DATA_DIR = os.environ.get("ROUTE_E_DATA", DATA_DIR)
GTF_PATH = os.environ.get("ROUTE_E_GTF", GTF_PATH)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
GDIR = os.path.join(DATA_DIR, GSE)
os.makedirs(GDIR, exist_ok=True)
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 16); pd.set_option("display.max_rows", 220)


class StopHere(Exception):
    """Stops the notebook with a plain-language message."""


print("Dataset:", GSE, "| folder:", GDIR)

In [ ]:
#@title Cell 2 - Download the single-cell files from GEO
BASE = os.environ.get("ROUTE_E_GEO", "https://ftp.ncbi.nlm.nih.gov/geo")


def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"   already here: {os.path.basename(dest)}"); return True
    r = subprocess.run(["curl", "-fsSL", "-m", "3600", "-o", dest + ".part", url], capture_output=True, text=True)
    if r.returncode != 0:
        if os.path.exists(dest + ".part"): os.remove(dest + ".part")
        return False
    os.replace(dest + ".part", dest)
    print(f"   downloaded {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.1f} MB)"); return True


def listing(url, tries=3):
    for _ in range(tries):
        r = subprocess.run(["curl", "-fsSL", "-m", "180", url], capture_output=True, text=True)
        if r.returncode == 0:
            return [h for h in re.findall(r'href="([^"?/][^"]*)"', r.stdout) if not h.startswith("http")]
    return []


def series_dir(acc):
    return f"{BASE}/series/{acc[:-3]}nnn/{acc}/"


def data_files(acc):
    """Supplementary files of a series: at series level, under its sub-series, or per sample."""
    sdir = series_dir(acc) + "suppl/"
    names = [h for h in listing(sdir) if not h.endswith("/") and h != "filelist.txt"]
    if names:
        return [(sdir + n, n) for n in names]
    out = []
    mdir = series_dir(acc) + "matrix/"
    for mn in [h for h in listing(mdir) if h.endswith("_series_matrix.txt.gz")]:
        mp = os.path.join(GDIR, mn)
        if not fetch(mdir + mn, mp):
            continue
        with gzip.open(mp, "rt", errors="replace") as h:
            head = h.read(400000)
        subs = re.findall(r"SuperSeries of: (GSE\d+)", head)
        for sub in dict.fromkeys(subs):                       # a SuperSeries keeps its files in the sub-series
            print(f"   {acc} is a SuperSeries; looking inside {sub}")
            out += data_files(sub)
        if not subs:
            gsms = dict.fromkeys(re.findall(r"GSM\d+", head))
            print(f"   no files at series level; checking {len(gsms)} samples one by one")
            for g in gsms:
                gd = f"{BASE}/samples/{g[:-3]}nnn/{g}/suppl/"
                out += [(gd + n, n) for n in listing(gd, tries=1) if not n.endswith("/") and n != "filelist.txt"]
    return out


EBI = os.environ.get("ROUTE_E_EBI", "https://ftp.ebi.ac.uk/pub/databases/microarray/data/atlas/sc_experiments")


def ebi_files(acc):
    """Some studies put only raw sequencing on GEO. EBI's Single Cell Expression Atlas holds the ready-made matrix."""
    e = "E-GEOD-" + re.sub(r"\D", "", acc)
    d = f"{EBI}/{e}/"
    names = [n for n in listing(d) if re.search(r"\.(mtx|mtx_rows|mtx_cols|tsv)$", n)]
    keep = [n for n in names if ".mtx" in n or "cell_metadata" in n or n.startswith("ExpDesign")]
    if keep:
        print(f"   GEO has no processed files; using EBI Single Cell Expression Atlas entry {e}")
    return [(d + n, n) for n in keep]


found = data_files(GSE) or ebi_files(GSE)
print(f"Files to download: {len(found)}")
for _, n in found[:12]:
    print("   ", n)
got = [os.path.join(GDIR, n) for u, n in found if fetch(u, os.path.join(GDIR, n))]
if not got:
    got = [os.path.join(GDIR, n) for n in sorted(os.listdir(GDIR))
           if not n.endswith((".part", ".csv", ".png")) and "series_matrix" not in n]
    if got:
        print("GEO did not answer; using files already in your folder:", [os.path.basename(g) for g in got])
if not got:
    raise StopHere(f"No data files found for {GSE} at the series, sub-series or sample level. "
                   "GEO may be busy: try running this cell again. If it keeps failing, send me this message.")
for p in [g for g in got if g.lower().endswith(".tar")]:        # unpack per-sample bundles
    with tarfile.open(p) as tf:
        tf.extractall(os.path.join(GDIR, "unpacked"))
    print(f"   unpacked {os.path.basename(p)}")
files = []
for root, _, names in os.walk(GDIR):
    files += [os.path.join(root, n) for n in names
              if not n.endswith((".part", ".tar")) and "series_matrix" not in n]
print(f"\n{len(files)} data files available.")

In [ ]:
#@title Cell 3 - Load the count matrices (cells x genes)
def read_mtx_set(mtx, bar, feat):
    """A 10x triplet: matrix.mtx(.gz), barcodes.tsv(.gz), features/genes.tsv(.gz)."""
    from scipy.io import mmread
    op = gzip.open if mtx.endswith(".gz") else open
    with op(mtx, "rb") as h:
        M = sp.csr_matrix(mmread(h))
    rd = lambda p: [l.rstrip("\n").split("\t") for l in (gzip.open(p, "rt") if p.endswith(".gz") else open(p))]
    cells_ = [r[0] for r in rd(bar)]
    rows = rd(feat)
    names = [r[1] if len(r) > 1 else r[0] for r in rows]           # column 2 is the gene symbol
    if M.shape[0] == len(names) and M.shape[1] == len(cells_):
        M = M.T.tocsr()
    return M, cells_, names


def read_dense(path):
    op = gzip.open(path, "rt") if path.endswith(".gz") else open(path)
    first = op.readline(); op.close()
    sep = "," if first.count(",") > first.count("\t") else "\t"
    d = pd.read_csv(path, sep=sep, index_col=0, low_memory=False)
    d = d.apply(pd.to_numeric, errors="coerce").dropna(how="all")
    if d.shape[0] < d.shape[1]:                                     # genes should be the long side
        d = d.T
    return sp.csr_matrix(d.to_numpy(dtype=np.float32).T), list(d.columns), list(d.index)


def to_symbols(gs):
    """EBI files name genes by FlyBase ID; turn them into gene names using the annotation."""
    if np.mean([str(g).startswith("FBgn") for g in gs[:500]]) <= 0.5:
        return list(gs), False
    if not os.path.exists(GTF_PATH):
        raise StopHere(f"The matrix uses FlyBase IDs and the annotation is not at {GTF_PATH}. It is the same file "
                       "used in Route B; fix GTF_PATH in Cell 1.")
    global _ID2SYM
    if "_ID2SYM" not in globals():
        _ID2SYM = {}
        with gzip.open(GTF_PATH, "rt") as h:
            for line in h:
                if "\tgene\t" not in line:
                    continue
                a = re.search(r'gene_id "([^"]+)"', line); b = re.search(r'gene_name "([^"]+)"', line)
                if a and b:
                    _ID2SYM[a.group(1)] = b.group(1)
    return [_ID2SYM.get(re.sub(r"\.\d+$", "", str(g)), str(g)) for g in gs], True


def rank_of(name):
    """Lower is better. GEO per-sample files keep the library structure, so they come first.
    The EBI entry ships the same data three times over; take the filtered raw counts."""
    n = name.lower()
    if "aggregated" in n or n.startswith("e-geod") or n.startswith("e-mtab"):
        if "filtered_normalised" in n:
            return 3
        return 1 if "filtered" in n else 2
    return 0


found_blocks = []
by_dir = {}
for p in files:
    by_dir.setdefault(os.path.dirname(p), []).append(p)
for d, ps in sorted(by_dir.items()):
    triplets = {}
    for p in ps:
        n = os.path.basename(p).lower()
        key = re.sub(r"((barcodes|features|genes|matrix)\.(tsv|mtx)|\.mtx(_rows|_cols)?)(\.gz)?$", "", n)
        kind = ("mtx" if n.endswith((".mtx", ".mtx.gz")) or ("matrix" in n and ".mtx" in n)
                else "bar" if "barcode" in n or n.endswith((".mtx_cols", ".mtx_cols.gz"))
                else "feat" if "feature" in n or "genes" in n or n.endswith((".mtx_rows", ".mtx_rows.gz"))
                else None)
        if kind:
            triplets.setdefault(key, {})[kind] = p
    for key, t in sorted(triplets.items()):
        if {"mtx", "bar", "feat"} <= set(t):
            M, cs, gs = read_mtx_set(t["mtx"], t["bar"], t["feat"])
            lib = (key.strip("_-. ") or os.path.basename(d)) or "library1"
            found_blocks.append({"rank": rank_of(lib), "lib": lib, "M": M, "genes": gs, "cells": cs})
if not found_blocks:                                                # no matrix triplets: try a dense table
    for p in sorted(files, key=os.path.getsize, reverse=True):
        if re.search(r"\.(csv|tsv|txt|tab)(\.gz)?$", p, re.I):
            try:
                M, cs, gs = read_dense(p)
            except Exception as ex:
                print(f"   {os.path.basename(p)}: could not read ({ex})"); continue
            if M.shape[0] > 200 and M.shape[1] > 2000:
                lib = re.sub(r"\.(csv|tsv|txt|tab)(\.gz)?$", "", os.path.basename(p), flags=re.I)
                found_blocks.append({"rank": rank_of(lib), "lib": lib, "M": M, "genes": gs, "cells": cs})
if not found_blocks:
    print("Files found:", [os.path.basename(f) for f in files][:40])
    raise StopHere("None of the files could be read as a single-cell matrix. Send me this output.")

print("Matrices found in your folder:")
for b in sorted(found_blocks, key=lambda b: b["rank"]):
    print(f"   {b['lib']}: {b['M'].shape[0]:,} cells x {b['M'].shape[1]:,} genes")
best = min(b["rank"] for b in found_blocks)
chosen = [b for b in found_blocks if b["rank"] == best]
if best > 0 and len(chosen) > 1:                                    # EBI ships one dataset in several forms
    chosen = chosen[:1]
if len(chosen) < len(found_blocks):
    print(f"\nSeveral versions of this dataset are present. Using "
          f"{'the per-sample files' if best == 0 else chosen[0]['lib']} and ignoring the rest, so that one dataset "
          "is not counted twice.")
blocks, groups = [], []
for b in chosen:
    gs, converted = to_symbols(b["genes"])
    blocks.append((b["M"], gs))
    groups.append(pd.DataFrame({"library": b["lib"], "barcode": b["cells"]}))
    print(f"Using {b['lib']}: {b['M'].shape[0]:,} cells x {b['M'].shape[1]:,} genes"
          + (" (FlyBase IDs converted to gene names)" if converted else ""))

shared = sorted(set.intersection(*[set(g) for _, g in blocks]))
if len(shared) < 3000:
    raise StopHere(f"Only {len(shared)} gene names are shared across the libraries. Send me this message.")
X = sp.vstack([M[:, [{g: i for i, g in enumerate(gs)}[s] for s in shared]] for M, gs in blocks]).tocsr()
cells = pd.concat(groups, ignore_index=True)
genes = pd.Index(shared)
X.data = np.nan_to_num(X.data)
cells["n_counts"] = np.asarray(X.sum(axis=1)).ravel()
cells["n_genes"] = np.asarray((X > 0).sum(axis=1)).ravel()
mito = [i for i, g in enumerate(genes) if g.startswith(("mt:", "mt-"))]
cells["pct_mito"] = 100 * np.asarray(X[:, mito].sum(axis=1)).ravel() / np.maximum(cells.n_counts, 1) if mito else 0.0
keep = (cells.n_genes >= MIN_GENES) & (cells.n_counts >= MIN_COUNTS) & (cells.pct_mito < 30)
print(f"\nCells before filtering: {len(cells):,}  |  kept: {int(keep.sum()):,}")
cells, X = cells[keep.to_numpy()].reset_index(drop=True), X[np.flatnonzero(keep.to_numpy())]
detected = np.asarray((X > 0).sum(axis=0)).ravel()
gkeep = detected >= 10
X, genes = X[:, np.flatnonzero(gkeep)].tocsr(), genes[gkeep]
print(f"Genes kept (detected in >=10 cells): {len(genes):,}  |  libraries: {cells.library.nunique()}")
display(cells.groupby("library").agg(cells=("barcode", "size"), median_counts=("n_counts", "median")))

In [ ]:
#@title Cell 4 - Normalise, cluster, and name the clusters from marker genes
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

gpos = {g: i for i, g in enumerate(genes)}
cpm = X.multiply(1e4 / np.maximum(cells.n_counts.to_numpy()[:, None], 1)).tocsr()
Ln = cpm.copy(); Ln.data = np.log1p(Ln.data)                       # log normalised expression per cell
v = np.asarray(Ln.power(2).mean(axis=0)).ravel() - np.asarray(Ln.mean(axis=0)).ravel() ** 2
hv = np.argsort(v)[-2000:]
Z = Ln[:, hv].toarray()
Z = (Z - Z.mean(0)) / np.where(Z.std(0) > 0, Z.std(0), 1)
svd = TruncatedSVD(n_components=30, random_state=SEED).fit_transform(np.clip(Z, -10, 10))
svd = svd / np.maximum(np.linalg.norm(svd, axis=1, keepdims=True), 1e-9)
cells["cluster"] = KMeans(n_clusters=N_CLUSTERS, n_init=5, random_state=SEED).fit_predict(svd).astype(str)


def score(gs):
    """Mean normalised expression of a gene set, per cell."""
    idx = [gpos[g] for g in gs if g in gpos]
    return np.asarray(Ln[:, idx].mean(axis=1)).ravel() if idx else np.zeros(len(cells)), len(idx)


MARKERS = {
    "follicle cell (somatic epithelium)": "eya br Fas3 CadN Fas2 Sox14 mirr pnt",
    "germline": "vas nos bam orb osk Rbp9 corolla",
    "escort / inner germarial sheath": "ptc hh tj Wnt4 bnl",
    "muscle sheath": "Mhc Mlc1 up zip Act57B",
    "stalk and polar cells": "upd1 LamC cas Fas3",
    "mitotic (cycling)": "stg CycB polo aurB Det pav PCNA CycA Incenp",
    "early follicle (stages 1-6)": "CadN Fas2 ct SPARC",
    "mid follicle (stages 7-10, endocycling)": "br mirr mid Yp1 Yp2 Yp3",
    "late follicle (stages 9+, endocycling)": "Sox14 Vha16-1 Pvf1 dpp cv-2",
    "vitelline membrane / chorion (amplifying)": "Vm26Aa Vm26Ab Vm32E Vm34Ca Cp36 Cp38 Cp15 Cp16 dec-1 psd",
}
tab = {}
for name, gs in MARKERS.items():
    s, n = score(gs.split())
    tab[f"{name} ({n})"] = pd.Series(s).groupby(cells.cluster.to_numpy()).mean()
tab = pd.DataFrame(tab)
zt = (tab - tab.mean()) / tab.std().replace(0, 1)
zt["cells"] = cells.cluster.value_counts()
display(zt.round(2).sort_values(zt.columns[0], ascending=False))
print("Columns are z-scores across clusters: a high value means that cluster expresses those markers most.")

In [ ]:
#@title Cell 5 - Pick mitotic and endocycling follicle cells, and check them
fc_col = [c for c in zt.columns if c.startswith("follicle cell")][0]
gl_col = [c for c in zt.columns if c.startswith("germline")][0]
mu_col = [c for c in zt.columns if c.startswith("muscle")][0]
mit_col = [c for c in zt.columns if c.startswith("mitotic")][0]
early_col = [c for c in zt.columns if c.startswith("early follicle")][0]
mid_col = [c for c in zt.columns if c.startswith("mid follicle")][0]
late_col = [c for c in zt.columns if c.startswith("late follicle")][0]
amp_col = [c for c in zt.columns if c.startswith("vitelline")][0]

esc_col = [c for c in zt.columns if c.startswith("escort")][0]
# Follicle cells are defined by EXCLUDING the other lineages. The general epithelial markers (CadN, Fas2, br)
# fade as follicle cells mature, so requiring them would throw away exactly the late, most polyploid cells.
is_fc = (zt[gl_col] < 0.5) & (zt[mu_col] < 0.5) & (zt[esc_col] < 0.5)
stage_raw = zt[[mid_col, late_col, amp_col]].mean(axis=1) - zt[early_col]
# Rank the follicle clusters among themselves, so the split is early vs late WITHIN the lineage
f = zt.index[is_fc]
stage = (stage_raw - stage_raw[f].mean()) / (stage_raw[f].std() or 1)
mitr = (zt[mit_col] - zt.loc[f, mit_col].mean()) / (zt.loc[f, mit_col].std() or 1)
role = {}
for cl in zt.index:
    if not is_fc[cl]:
        role[cl] = "other cell type"
    elif mitr[cl] > 0 and stage[cl] < 0:
        role[cl] = "mitotic follicle cell"
    elif mitr[cl] < 0 and stage[cl] > 0:
        role[cl] = "endocycling follicle cell"
    else:
        role[cl] = "follicle cell, unclear stage"
cells["role"] = cells.cluster.map(role)
summary = pd.DataFrame({"role": pd.Series(role), "mitotic_z": mitr.round(2),
                        "stage_score": stage.round(2), "cells": cells.cluster.value_counts()})
display(summary.sort_values(["role", "stage_score"]))
print(cells.role.value_counts().to_string())
MIT, ENDO = "mitotic follicle cell", "endocycling follicle cell"
if not {MIT, ENDO} <= set(cells.role.unique()):
    raise StopHere("Could not find both mitotic and endocycling follicle cell clusters. Send me the tables above; "
                   "raising N_CLUSTERS in Cell 1 often separates them.")

# Checks: the two groups must differ in the expected way
for name, gs in [("mitotic genes (expected much lower in endocycling cells)", MARKERS["mitotic (cycling)"]),
                 ("early markers CadN, Fas2, ct (expected lower in endocycling cells)", MARKERS["early follicle (stages 1-6)"]),
                 ("yolk and mid markers (expected higher in endocycling cells)", MARKERS["mid follicle (stages 7-10, endocycling)"])]:
    s, n = score(gs.split())
    a, b = s[(cells.role == MIT).to_numpy()].mean(), s[(cells.role == ENDO).to_numpy()].mean()
    print(f"{name}: mitotic {a:.2f}, endocycling {b:.2f}, difference {b - a:+.2f}")
print(f"\nMedian counts per cell: mitotic {cells.n_counts[cells.role == MIT].median():,.0f}, "
      f"endocycling {cells.n_counts[cells.role == ENDO].median():,.0f}")

# Dissociation stress: pulling a tissue apart switches on heat shock and immediate-early genes, and it hits
# large fragile cells hardest. If one group is more stressed, that alone shifts many genes.
DISSOC = "Hsp70Aa Hsp70Ab Hsp68 Hsp26 Hsp23 Hsp27 Hsp83 DnaJ-1 Hsf Jra kay Hr38 sr puc".split()
cells["stress"], _ = score(DISSOC)[0], None
cells["stress"] = score(DISSOC)[0]
for r in (MIT, ENDO):
    v = cells.stress[cells.role == r]
    print(f"Dissociation stress score, {r}: median {v.median():.2f}")
cut = min(np.quantile(cells.stress[cells.role == MIT], 0.75), np.quantile(cells.stress[cells.role == ENDO], 0.75))
cells["low_stress"] = cells.stress <= cut
print(f"Cells kept for the stress-matched comparison (score <= {cut:.2f}): "
      f"{int(((cells.role == MIT) & cells.low_stress).sum()):,} mitotic, "
      f"{int(((cells.role == ENDO) & cells.low_stress).sum()):,} endocycling")

In [ ]:
#@title Cell 6 - Redox programmes: endocycling vs mitotic follicle cells
PROGRAMMES = {
    "REDOX: glutathione S-transferases": "GstD1 GstD2 GstD3 GstD5 GstD9 GstE1 GstE3 GstE6 GstE7 GstE9 GstE12 GstE14 GstS1 GstO1 GstO2 GstO3 GstT1 GstZ1 GstZ2 Mgstl",
    "REDOX: glutathione synthesis & recycling": "Gclc Gclm Gss1 Gss2 Gr1 Grx1 Grx1t PHGPx GILT1",
    "REDOX: thioredoxin / peroxiredoxin": "Trxr-1 Trxr-2 Trx-2 TrxT dhd Jafrac1 Jafrac2 Prx3 Prx5 Prx2540-1 Prx2540-2 Srx",
    "REDOX: SOD and catalase": "Sod1 Sod2 Sod3 Cat",
    "REDOX: NADPH supply": "Zw Pgd Men Men-b Idh Idh3a Nmdmc Tkt Taldo",
    "REDOX: cnc (NRF2) targets": "cnc Keap1 GstD1 Gclc Gclm Cyp6a2 Cyp6a8 Ugt35a Jafrac1 Prx3",
    "REDOX: H2O2 / ROS-producing enzymes": "Ero1L Duox Nox Cyp4e3 Mav Uro Aox1 Pxd Pxt",
    "REDOX: mitochondrial respiration": "ND-75 ND-49 ND-30 ND-42 SdhA SdhB UQCR-C1 UQCR-C2 Cyt-c-p COX4 COX5A ATPsynbeta blw",
    "EXPLORE: ER folding load": "Xbp1 Hsc70-3 Ire1 PEK crc Pdi ERp60 Calr Ero1L Sec61alpha Sec61beta Srp54",
    "EXPLORE: ribosome / protein synthesis": "RpL3 RpL4 RpL5 RpL7 RpL8 RpL10 RpL11 RpL13 RpS2 RpS3 RpS4 RpS6 RpS8 RpS9 RpS11 RpS15",
    "EXPLORE: licensing and re-replication": "geminin dup Cdc6 Orc1 Mcm2 Mcm3 dpa Mcm5 Mcm6 Mcm7 CycE E2f1 Myc",
    "EXPLORE: p53 / DNA damage": "p53 Ku80 Irbp lok mei-41 tefu rad50 Xrp1 spn-A mus304",
    "CONTROL: heat shock": "Hsp70Aa Hsp83 Hsp68 Hsp27 Hsp26 Hsp23 DnaJ-1 Hsf",
    # CrebA was removed from this control set. It is the master regulator of secretory capacity in flies, so in a
    # tissue that is becoming secretory it is not a generic-stress gene at all; leaving it in made the stress
    # control move for a biological reason and so weakened every comparison judged against it.
    "CONTROL: immediate-early genes": "Hr38 sr kay Jra puc",
    "CONTROL: housekeeping": "Act5C alphaTub84B RpL32 Gapdh1 Tbp eIF4A Pgk Ef1alpha48D",
}
# Some genes are listed under a different name in the annotation, so each one is looked up under its synonyms too.
# FlyBase calls Geminin "geminin" (CG3183); "Gem" is only a synonym, which is why earlier runs could not find it.
ALIASES = {"geminin": ["Gem", "CG3183"], "dup": ["Cdt1"], "dpa": ["Mcm4"], "Zw": ["G6PD", "G6pd"],
           "Ero1L": ["Ero1", "CG1333"], "PHGPx": ["CG12013"], "Gss1": ["Gss"], "Gr1": ["Gr", "GR"],
           "Trxr-1": ["Trxr1"], "Jafrac1": ["Prx2"], "Men-b": ["Men-b", "CG5889"]}


def resolve(g, present):
    if g in present:
        return g
    for a in ALIASES.get(g, []):
        if a in present:
            return a
    return None


_renamed = {}
_resolved = {}
for n, v in PROGRAMMES.items():
    out = []
    for g in v.split():
        r = resolve(g, gpos)
        if r:
            out.append(r)
            if r != g:
                _renamed[g] = r
    _resolved[n] = out
PROGRAMMES = _resolved
if _renamed:
    print("Found under a synonym:", ", ".join(f"{k} -> {v}" for k, v in _renamed.items()))
_lic = [n for n in PROGRAMMES if "licensing" in n]
if _lic:
    print("Licensing set as tested:", ", ".join(PROGRAMMES[_lic[0]]))
for n, v in PROGRAMMES.items():
    print(f"{n}: {len(v)} genes found")
PROGRAMMES = {n: v for n, v in PROGRAMMES.items() if len(v) >= 3}

# Pseudobulk: add up the counts of each group, per library, then normalise by median of ratios.
# This avoids yolk protein genes (very high in endocycling cells) dominating the scale.
pb, lab = [], []
for lib in sorted(cells.library.unique()):
    for r in (MIT, ENDO):
        m = ((cells.library == lib) & (cells.role == r) & cells.low_stress).to_numpy()
        if m.sum() >= 50:
            pb.append(np.asarray(X[np.flatnonzero(m)].sum(axis=0)).ravel()); lab.append((lib, r))
pbd = pd.DataFrame(np.array(pb).T, index=genes, columns=pd.MultiIndex.from_tuples(lab))
pbd = pbd[(pbd / pbd.sum() * 1e6 >= 5).all(axis=1)]
lg = np.log(pbd.replace(0, np.nan)); ref = lg.mean(axis=1)
size = np.exp(lg[ref.notna()].sub(ref[ref.notna()], axis=0).median())
Lp = np.log2(pbd / size / pbd.sum().mean() * 1e6 + 1)
n_lib = len({l for l, _ in lab if (l, MIT) in lab and (l, ENDO) in lab})
print(f"\nStress-matched cells only. Pseudobulk groups: {len(lab)} ({n_lib} librar{'y' if n_lib == 1 else 'ies'} have both cell groups), "
      f"{len(Lp):,} genes expressed in all of them")
fc = Lp.xs(ENDO, level=1, axis=1).mean(axis=1) - Lp.xs(MIT, level=1, axis=1).mean(axis=1)
per_lib = {l: Lp[(l, ENDO)] - Lp[(l, MIT)] for l in sorted({a for a, b in lab}) if (l, ENDO) in lab and (l, MIT) in lab}

level = Lp.mean(axis=1)
bins = pd.qcut(level.rank(method="first"), 10, labels=False)
avoid = {g for v in PROGRAMMES.values() for g in v}
pools = {b: [g for g in bins.index[bins == b] if g not in avoid] for b in range(10)}
controls = [n for n in PROGRAMMES if n.startswith("CONTROL")]
rows = []
for n, gs in PROGRAMMES.items():
    gs = [g for g in gs if g in fc.index]
    if len(gs) < 3:
        continue
    obs = fc.loc[gs].median()
    null = np.array([fc.loc[[rng.choice(pools[bins[g]]) for g in gs]].median() for _ in range(N_RANDOM_SETS)])
    c = np.median(null)
    r = {"programme": n, "genes": len(gs), "log2_change": obs, "random_median": c,
         "p_vs_random": (1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (N_RANDOM_SETS + 1),
         "beyond_all_controls": all(np.sign(obs - fc.loc[[x for x in PROGRAMMES[k] if x in fc.index]].median())
                                    == np.sign(obs - c) for k in controls)}
    for l, d in per_lib.items():
        r[f"lib_{l[:14]}"] = d.loc[gs].median()
    rows.append(r)
res = pd.DataFrame(rows)
m = ~res.programme.str.startswith("CONTROL")
p = res.loc[m, "p_vs_random"].to_numpy(); o = np.argsort(p)
q = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
qq = np.empty_like(p); qq[o] = np.minimum(q, 1); res.loc[m, "q_fdr"] = qq
display(res.round(3))
print("log2_change: endocycling minus mitotic follicle cells. Positive means higher in endocycling cells.")
libcols = [c for c in res.columns if c.startswith("lib_")]
print("\nVerdict:")
for _, r in res[m].iterrows():
    agree = len(libcols) < 2 or len({np.sign(r[c] - res.loc[m, c].median()) for c in libcols}) == 1
    v = ("HIGHER" if r.log2_change > r.random_median else "LOWER") + " in endocycling follicle cells"
    if r.q_fdr >= 0.05:
        v = "no difference from random genes"
    elif not r.beyond_all_controls:
        v += " (but not beyond every stress/housekeeping control)"
    elif not agree:
        v += " (but the libraries disagree)"
    print(f"   {r.programme}: {v} ({r.log2_change:+.2f})")
if len(libcols) < 2:
    print("\nOnly one library has both cell groups, so there is no biological replication. The comparison with random "
          "gene sets is the only safeguard here: read the result as a lead, not as a measured effect size.")
print("\nNote: in the follicle cell lineage, ploidy and developmental stage rise together. A difference here means "
      "endocycling cells differ from mitotic ones, not that ploidy alone caused it.")

In [ ]:
#@title Cell 7 - Gene by gene, the graded stage trend, chart and saving
gl = list(dict.fromkeys(g for n, v in PROGRAMMES.items() if not n.startswith("CONTROL") for g in v if g in fc.index))
tab2 = pd.DataFrame({"log2_change": fc.loc[gl], "pct_of_all_genes": fc.rank(pct=True).loc[gl],
                     "mean_mitotic": Lp.xs(MIT, level=1, axis=1).mean(axis=1).loc[gl],
                     "mean_endocycling": Lp.xs(ENDO, level=1, axis=1).mean(axis=1).loc[gl]})
for l, d in per_lib.items():
    tab2[f"lib_{l[:14]}"] = d.loc[gl]
tab2["programme"] = [", ".join(n.split(": ")[1] for n, v in PROGRAMMES.items() if g in v) for g in gl]
display(tab2.sort_values("log2_change").round(2))

# Graded check: do the programmes change step by step across the follicle cell clusters, ordered by stage?
order = [cl for cl in summary.index if summary.loc[cl, "role"].startswith("follicle cell") or
         summary.loc[cl, "role"].endswith("follicle cell")]
order = sorted(order, key=lambda cl: stage[cl])
grade = {}
for n, gs in PROGRAMMES.items():
    s, _ = score(gs)
    grade[n.split(": ")[1]] = pd.Series(s).groupby(cells.cluster.to_numpy()).mean().reindex(order)
grade = pd.DataFrame(grade).T
grade.columns = [f"c{cl} ({summary.loc[cl, 'role'].replace(' follicle cell', '')}, stage {stage[cl]:+.1f})" for cl in order]
display(grade.round(2))
print("Follicle cell clusters left to right by stage score (earliest first). Values are mean log expression per cell.")

# Licensing genes one by one, in stage order: does Geminin change before or after the switch?
lic_name = next((n for n in PROGRAMMES if "licensing" in n), None)
if lic_name:
    lic = PROGRAMMES[lic_name]
    rows_l = {}
    for g in lic:
        v = np.asarray(Ln[:, gpos[g]].todense()).ravel()
        rows_l[g] = pd.Series(v).groupby(cells.cluster.to_numpy()).mean().reindex(order)
    lic_tab = pd.DataFrame(rows_l).T
    lic_tab.columns = grade.columns
    switch = [i for i, cl in enumerate(order) if summary.loc[cl, "role"] == ENDO]
    first_endo = min(switch) if switch else None
    print(f"\nLicensing genes across the follicle clusters, earliest stage first "
          f"({lic_name.split(': ')[1]}):")
    display(lic_tab.round(2))
    if first_endo is not None:
        print(f"The switch sits between column {first_endo - 1} and {first_endo}: columns 0-{first_endo - 1} are "
              f"mitotic clusters, {first_endo} onwards are endocycling.")

    print("\nEndocycling minus mitotic, per gene (same numbers as the table above, summarised):")
    lic_sum = pd.DataFrame({"log2_change": fc.reindex(lic),
                            "pct_of_all_genes": fc.rank(pct=True).reindex(lic)})
    for l, d in per_lib.items():
        lic_sum[f"lib_{l[:14]}"] = d.reindex(lic)
    libc = [c for c in lic_sum.columns if c.startswith("lib_")]
    lic_sum["same_direction"] = (np.sign(lic_sum[libc]).nunique(axis=1) == 1) if len(libc) > 1 else True
    display(lic_sum.sort_values("log2_change").round(2))
    for g in ("geminin", "Gem", "dup", "Cdt1"):
        if g in fc.index:
            d = fc[g]
            print(f"   {g}: {d:+.2f} log2 ({'lower' if d < 0 else 'higher'} in endocycling cells, "
                  f"percentile {fc.rank(pct=True)[g]:.2f} of all genes)")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for g in lic_tab.index:
        key = g in ("geminin", "Gem")
        ax.plot(range(lic_tab.shape[1]), lic_tab.loc[g], marker="o", ms=4,
                lw=2.5 if key else 1, color="#b2432f" if key else "#9aa7b4",
                label=g if key or lic_tab.loc[g].std() > 0.3 else None, zorder=3 if key else 1)
    if first_endo is not None:
        ax.axvline(first_endo - 0.5, color="black", ls="--", lw=1)
        ax.text(first_endo - 0.45, ax.get_ylim()[1], " switch", va="top", fontsize=8)
    ax.set_xticks(range(lic_tab.shape[1])); ax.set_xticklabels([c.split(" (")[0] for c in lic_tab.columns], fontsize=8)
    ax.set_xlabel("follicle cell clusters, earliest stage first"); ax.set_ylabel("mean log expression per cell")
    ax.set_title("Licensing genes across the mitotic-to-endocycle switch", loc="left", fontsize=10)
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(GDIR, "chart_licensing.png"), dpi=150); plt.show()
    lic_tab.to_csv(os.path.join(GDIR, "licensing_by_stage.csv"))

fig, ax = plt.subplots(figsize=(8, 0.34 * len(res) + 1.4))
col = ["#b2432f" if v < 0 else "#2f6db2" for v in (res.log2_change - res.random_median)]
ax.barh(res.programme, res.log2_change - res.random_median, color=col)
for i, r in res.reset_index().iterrows():
    ax.text(r.log2_change - r.random_median, i, f"  q={r.q_fdr:.3f}" if r.q_fdr == r.q_fdr else "  (control)",
            va="center", fontsize=8)
ax.axvline(0, color="black", lw=0.8); ax.invert_yaxis()
ax.set_xlabel("log2 change, endocycling vs mitotic follicle cells, minus random genes")
ax.set_title(f"{GSE}: programmes across the mitotic-to-endocycle switch", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(GDIR, "chart_programmes.png"), dpi=150); plt.show()
res.to_csv(os.path.join(GDIR, "programmes.csv"), index=False)
tab2.to_csv(os.path.join(GDIR, "genes.csv")); grade.to_csv(os.path.join(GDIR, "stage_trend.csv"))
cells[["library", "barcode", "cluster", "role", "n_counts", "n_genes"]].to_csv(os.path.join(GDIR, "cells.csv"), index=False)
print("Saved to", GDIR)

In [ ]:
#@title Cell 8 - How much is there? Absolute abundance of the redox enzymes
PRESENT = gpos
OUTDIR = GDIR
GROUPS = [MIT, ENDO]
LEVELS = pd.DataFrame({MIT: Lp.xs(MIT, level=1, axis=1).mean(axis=1),
                       ENDO: Lp.xs(ENDO, level=1, axis=1).mean(axis=1)})
# The screens above measure CHANGE. A programme can shift a long way and still be a minor part
# of the cell, and a gene can be hugely abundant and barely move. This cell reports amount.
CLASSES = {
    "H2O2 removal": "Jafrac1 Jafrac2 Prx3 Prx5 Prx2540-1 Prx2540-2 PHGPx Cat Srx",
    "thioredoxin system (regenerates peroxiredoxins)": "Trxr-1 Trxr-2 Trx-2 TrxT dhd",
    "superoxide dismutases": "Sod1 Sod2 Sod3",
    "glutathione supply": "Gclc Gclm Gss1 Gss2 Gr1 Grx1 Grx1t GILT1",
    "NADPH supply": "Zw Pgd Men Men-b Idh Idh3a Tkt Taldo Nmdmc",
    "H2O2 sources": "Ero1L Duox Nox Aox1 Pxd Pxt Uro Mav Cyp4e3",
}
CLASSES = {k: [r for g in v.split() if (r := resolve(g, PRESENT)) and r in LEVELS.index]
           for k, v in CLASSES.items()}
pct_level = LEVELS.mean(axis=1).rank(pct=True)
rows = []
for cls, gs in CLASSES.items():
    for g in gs:
        r = {"gene": g, "class": cls}
        for grp in GROUPS:
            r[grp] = LEVELS[grp].loc[g] if g in LEVELS.index else np.nan
        r["abundance_pct"] = pct_level.get(g, np.nan)
        rows.append(r)
ab = pd.DataFrame(rows).dropna(subset=GROUPS, how="all").set_index("gene")
ab["change"] = ab[GROUPS[-1]] - ab[GROUPS[0]]
display(ab.sort_values(["class", GROUPS[0]], ascending=[True, False]).round(2))
print("Values are log2 CPM. abundance_pct is the gene's rank by expression among all "
      f"{len(pct_level):,} expressed genes (1.00 = highest).")

print("\nTotal capacity per class, as CPM summed across its genes (not logs, so these are comparable amounts):")
tot = pd.DataFrame({grp: ab.groupby("class")[grp].apply(lambda s: (2 ** s.dropna()).sum()) for grp in GROUPS})
tot["fold_change"] = tot[GROUPS[-1]] / tot[GROUPS[0]]
display(tot.round(1))
rm, sr = "H2O2 removal", "H2O2 sources"
if rm in tot.index and sr in tot.index:
    for grp in GROUPS:
        print(f"   {grp}: H2O2-removing transcripts are {tot.loc[rm, grp] / max(tot.loc[sr, grp], 1e-9):.0f}x "
              f"the H2O2-producing ones")

fig, ax = plt.subplots(figsize=(7.5, 0.26 * len(ab) + 1.6))
ab2 = ab.sort_values(["class", GROUPS[0]])
y = np.arange(len(ab2))
for k, grp in enumerate(GROUPS):
    ax.barh(y + (k - (len(GROUPS) - 1) / 2) * 0.38, 2 ** ab2[grp].to_numpy(), height=0.36, label=grp)
ax.set_yticks(y); ax.set_yticklabels([f"{g}  ({c.split(' (')[0]})" for g, c in zip(ab2.index, ab2["class"])], fontsize=7)
ax.set_xscale("log"); ax.set_xlabel("CPM (log scale)"); ax.invert_yaxis(); ax.legend(fontsize=8)
ax.set_title("How much of each redox enzyme is present", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR, "chart_abundance.png"), dpi=150); plt.show()
ab.to_csv(os.path.join(OUTDIR, "abundance.csv")); tot.to_csv(os.path.join(OUTDIR, "abundance_totals.csv"))
print("\nTranscript amount is not protein amount and not enzymatic flux. Peroxiredoxins are consumed per molecule of "
      "H2O2 and must be regenerated by thioredoxin and NADPH, while Ero1 turns over catalytically, so an abundant "
      "sink does not automatically outweigh a scarce source.")

In [ ]:
#@title Cell 9 - Compartment-resolved redox screen, including the nucleus
# The sets above pool enzymes that work in different places. Jafrac1 (cytosol) and Jafrac2 (ER)
# sit in one "thioredoxin / peroxiredoxin" set and move differently, which hides the compartment
# where H2O2 is actually being handled. This cell splits every redox gene by where it works and
# adds the nucleus, which no earlier set covered.
#
# Nuclear H2O2 in flies: Su(var)3-3 is the fly LSD1 / KDM1A, an FAD-dependent H3K4 demethylase
# that releases one H2O2 per demethylation, on chromatin. Flies have no NOX4, so the other
# nuclear source described in mammals is not available here. Rrp1 is the fly APEX1, whose repair
# role is specific to oxidative DNA damage; dhd and TrxT are nuclear-acting thioredoxins.
# Note also that the perinuclear space is continuous with the ER lumen and H2O2 crosses the
# nuclear pore freely, so ER-generated H2O2 is topologically next door to the nucleoplasm.
ALIASES.update({
    "Su(var)3-3": ["dLsd1", "Lsd1", "Su-var(3)3", "Suvar(3)3", "SuVar3-3", "CG17149"],
    "Rrp1": ["APEX1", "Rrp-1", "CG3178"],
    "Ogg1": ["dOgg1", "CG1795"],
})

COMPART = {
    ("nucleus", "H2O2 source (chromatin demethylation)"): "Su(var)3-3",
    ("nucleus", "thioredoxins and oxidative DNA repair"): "dhd TrxT Rrp1 Ogg1",
    ("ER", "H2O2 source"): "Ero1L",
    ("ER", "H2O2 removal"): "Jafrac2",
    ("ER", "folding load"): "Hsc70-3 Pdi ERp60 Calr Sec61alpha Sec61beta Xbp1 Ire1 PEK crc Srp54",
    ("cytosol", "H2O2 removal"): "Jafrac1 Prx2540-1 Prx2540-2 PHGPx",
    ("cytosol", "thioredoxin system"): "Trxr-1 Trx-2 Srx",
    ("cytosol", "glutathione supply"): "Gclc Gclm Gss1 Gss2 Gr1 Grx1 Grx1t",
    ("cytosol", "superoxide dismutase"): "Sod1 Sod3",
    ("mitochondria", "H2O2 removal"): "Prx3 Prx5 Trxr-2",
    ("mitochondria", "superoxide dismutase"): "Sod2",
    ("peroxisome", "H2O2 removal"): "Cat",
    ("plasma membrane", "H2O2 source"): "Duox Nox",
}

pd.set_option("display.max_columns", None, "display.width", 250)
present = set(fc.index)
resolved, missing = {}, []
for (comp, role), v in COMPART.items():
    keep = []
    for g in v.split():
        r = resolve(g, present)
        if r:
            keep.append(r)
        else:
            missing.append(g)
    resolved[(comp, role)] = keep
if missing:
    print("Not detected in this dataset:", ", ".join(missing))

pct = fc.rank(pct=True)
# A gene can read "flat" for three different reasons, and they are not interchangeable:
#   it really does not change; it is already near the top of the expression range and cannot rise
#   much further; or it is barely detected and the estimate is noise pulled toward the middle.
# Report expression level and detection rate so the three can be told apart.
expr_pct = level.rank(pct=True)
_mit = np.flatnonzero(((cells.role == MIT) & cells.low_stress).to_numpy())
_end = np.flatnonzero(((cells.role == ENDO) & cells.low_stress).to_numpy())
_all = sorted({g for gs in resolved.values() for g in gs})
_idx = [gpos[g] for g in _all]
_dm = np.asarray((X[_mit][:, _idx] > 0).mean(axis=0)).ravel()
_de = np.asarray((X[_end][:, _idx] > 0).mean(axis=0)).ravel()
detect = {g: (float(a), float(b)) for g, a, b in zip(_all, _dm, _de)}

rows = []
for (comp, role), gs in resolved.items():
    for g in gs:
        dm, de = detect[g]
        r = {"gene": g, "compartment": comp, "role": role,
             "log2_change": float(fc.loc[g]), "pct_of_all_genes": float(pct.loc[g]) * 100,
             "expression_pct": float(expr_pct.loc[g]) * 100,
             "detected_mitotic_%": round(100 * dm, 1), "detected_endocycling_%": round(100 * de, 1),
             "verdict": ("too sparse to judge" if max(dm, de) < 0.10 else
                         "at ceiling" if expr_pct.loc[g] > 0.95 and abs(fc.loc[g]) < 0.3 else
                         "measurable")}
        for l, d in per_lib.items():
            r[f"lib_{l[:14]}"] = float(d.loc[g])
        rows.append(r)
gene_tab = pd.DataFrame(rows).set_index("gene")
print("\nEvery redox gene by where it works. Negative = lower in endocycling cells.")
print("'too sparse to judge' means under 10% of cells detect it in both groups, so a flat value there is "
      "an absence of measurement, not an absence of change.")
display(gene_tab.sort_values(["compartment", "role", "log2_change"]).round(2))

# Per compartment and role: does the group move more than random genes matched on expression?
rows = []
for (comp, role), gs in resolved.items():
    if not gs:
        continue
    obs = float(fc.loc[gs].median())
    null = np.array([fc.loc[[rng.choice(pools[bins[g]]) for g in gs]].median() for _ in range(N_RANDOM_SETS)])
    c = float(np.median(null))
    p = (1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (N_RANDOM_SETS + 1) if len(gs) >= 3 else np.nan
    rows.append({"compartment": comp, "role": role, "genes": len(gs), "log2_change": obs,
                 "random_median": c, "vs_random": obs - c,
                 "p_vs_random": p, "median_pct": float(pct.loc[gs].median()) * 100})
comp_tab = pd.DataFrame(rows).sort_values(["compartment", "role"])
print("\nBy compartment. p is only computed where a group has at least three detected genes; "
      "for smaller groups read the percentile.")
display(comp_tab.round(3))

# The question the pooled sets could not answer: within one compartment, do source and sink move together?
print("\n--- Source against sink, within each compartment ---")
for comp in ["ER", "nucleus", "cytosol", "mitochondria"]:
    src = [g for (c, r), gs in resolved.items() if c == comp and "source" in r for g in gs]
    snk = [g for (c, r), gs in resolved.items() if c == comp and ("removal" in r or "thioredoxin" in r) for g in gs]
    if not src or not snk:
        print(f"{comp}: not enough detected genes on both sides "
              f"({len(src)} source, {len(snk)} removal) - cannot compare")
        continue
    # Judge each side against the genome, not against zero: a whole library can drift, and a
    # +0.04 is flat however positive its sign.
    s, k = float(fc.loc[src].median()), float(fc.loc[snk].median())
    sp, kp = float(pct.loc[src].median()) * 100, float(pct.loc[snk].median()) * 100
    word = lambda p: "up" if p > 65 else "down" if p < 35 else "flat"
    print(f"{comp}: sources {s:+.2f} ({sp:.0f}th pct, {word(sp)}) [{', '.join(src)}]; "
          f"removal {k:+.2f} ({kp:.0f}th pct, {word(kp)}) [{', '.join(snk)}]")
    if word(sp) == "up" and word(kp) != "up":
        print(f"    Production rises and removal does not follow: an imbalance local to the {comp}.")
    elif word(sp) == "up" and word(kp) == "up":
        print("    Both rise together, so capacity is tracking production.")
    elif word(sp) != "up" and word(kp) == "down":
        print(f"    Removal falls without a rise in production: less reducing capacity in the {comp}.")
    else:
        print("    Neither side moves clearly against the genome.")

ercmp = [g for (c, r), gs in resolved.items() if c == "ER" and "folding" in r for g in gs]
if ercmp and resolved.get(("ER", "H2O2 removal")):
    j2 = resolved[("ER", "H2O2 removal")][0]
    print(f"\nER folding load median {fc.loc[ercmp].median():+.2f} "
          f"({pct.loc[ercmp].median() * 100:.0f}th percentile) against {j2} "
          f"{fc.loc[j2]:+.2f} ({pct.loc[j2] * 100:.0f}th percentile).")
    print("The perinuclear space is continuous with the ER lumen, so an ER imbalance is the most "
          "likely route to a nuclear H2O2 signal in a secretory tissue.")

gene_tab.to_csv(os.path.join(GDIR, "compartments_genes.csv"))
comp_tab.to_csv(os.path.join(GDIR, "compartments_summary.csv"), index=False)
print("\nSaved compartments_genes.csv and compartments_summary.csv to", GDIR)

In [ ]:
#@title Cell 10 - The same switch in the germline: nurse cells as an independent endocycling lineage
# Follicle cells endocycle AND differentiate into an eggshell-secreting epithelium at the same time, so this
# notebook cannot tell those apart. The germline can. Cystocytes divide mitotically in the germarium, then
# become endocycling nurse cells - the same switch, in the same egg chambers, in the same libraries, with a
# completely different differentiation programme (maternal mRNA and protein, no eggshell).
#   Change seen in BOTH lineages  -> tracks endocycling.
#   Change seen only in follicle cells -> tracks secretory differentiation.
fc_follicle = fc.copy()
res_follicle = res.copy()

gl_cells = cells.cluster.map(zt[gl_col] >= 0.5).fillna(False).to_numpy()
if gl_cells.sum() < 100:
    raise StopHere(f"Only {int(gl_cells.sum())} germline cells found, too few for this comparison. "
                   "The germline marker column is " + repr(gl_col) + "; check the z-score table in Cell 4.")
g_clusters = [cl for cl in zt.index if zt.loc[cl, gl_col] >= 0.5]
print(f"Germline clusters: {g_clusters}  ({int(gl_cells.sum()):,} cells)")

# Split the germline on M PHASE alone. Nurse cells endocycle, so they run S phase repeatedly and express
# PCNA and the Mcm genes; what they never do is divide. So S-phase genes cannot tell the two apart and
# M-phase and cytokinesis genes can. (The follicle-cell marker set mixes the two, which is why it was wrong
# here.) bam marks germarial cysts, but it is on in only a few cells, so it is a check and not a requirement:
# vas, nos and Sxl are pan-germline and are higher in nurse cells simply because those cells are enormous.
MPHASE = "polo aurB pav Incenp CycB Det Sak Cks30A".split()
EARLY_GL = "bam bgcn".split()
#@markdown Leave empty to split automatically. If the automatic split looks wrong in the table below, put the
#@markdown germarium cluster ids here, comma separated, and run this cell again.
GERMARIUM_CLUSTERS = ""  #@param {type:"string"}

# When clustering resolves only one germline population there is nothing to split by cluster. Fall back to
# ranking the germline CELLS themselves by M-phase score and comparing the top and bottom thirds. This is
# weaker than an independent split - the groups are defined by a score rather than found - but the redox
# genes are not in that score, and the same strict control still has to pass for anything to be read.
PER_CELL_SPLIT = len(g_clusters) < 2
if PER_CELL_SPLIT:
    print(f"\nOnly one germline cluster, so splitting the {int(gl_cells.sum()):,} germline cells themselves "
          "by M-phase score (top third against bottom third). Weaker than a cluster split: read the control "
          "below before anything else.")
    mcell = pd.Series(score(MPHASE)[0])[gl_cells]
    hi, lo = np.quantile(mcell, [2 / 3, 1 / 3])
    cells["gl_role"] = None
    cells.loc[gl_cells & (pd.Series(score(MPHASE)[0]) >= hi).to_numpy(), "gl_role"] = "mitotic germline (germarium)"
    cells.loc[gl_cells & (pd.Series(score(MPHASE)[0]) <= lo).to_numpy(), "gl_role"] = "endocycling germline (nurse cells)"
    display(cells[cells.gl_role.notna()].groupby("gl_role").agg(
        cells=("gl_role", "size"), median_counts=("n_counts", "median")).round(0))

gm = pd.Series(score(MPHASE)[0]).groupby(cells.cluster.to_numpy()).mean().reindex(g_clusters)
ge = pd.Series(score(EARLY_GL)[0]).groupby(cells.cluster.to_numpy()).mean().reindex(g_clusters)
n_counts = cells.groupby("cluster").n_counts.median().reindex(g_clusters)
zm = (gm - gm.mean()) / (gm.std() or 1)
ze = (ge - ge.mean()) / (ge.std() or 1)

forced = [c.strip() for c in GERMARIUM_CLUSTERS.split(",") if c.strip()]
if PER_CELL_SPLIT:
    grole = {}
elif forced:
    grole = {cl: ("mitotic germline (germarium)" if cl in forced else "endocycling germline (nurse cells)")
             for cl in g_clusters}
    print("Using the clusters you named as germarium:", forced)
else:
    # Rank on M phase and cut at the largest gap, which puts the dividing clusters on one side however many
    # clusters there are. With two clusters this is simply the split between them.
    srt = gm.sort_values()
    gaps = srt.diff().iloc[1:]
    cutm = srt.loc[gaps.idxmax()] if len(srt) > 1 and gaps.notna().any() else gm.median()
    grole = {cl: ("mitotic germline (germarium)" if gm[cl] >= cutm else "endocycling germline (nurse cells)")
             for cl in g_clusters}
    if len(g_clusters) > 2 and (gm.max() - gm.min()) < 1e-9:
        raise StopHere("The germline clusters do not differ in M-phase gene expression at all, so they cannot "
                       "be split. Raise N_CLUSTERS in Cell 1.")
if not PER_CELL_SPLIT:
    display(pd.DataFrame({"role": pd.Series(grole), "Mphase_z": zm.round(2), "bam_z": ze.round(2),
                          "median_counts": n_counts.round(0),
                          "cells": cells.cluster.value_counts().reindex(g_clusters)}))
    cells["gl_role"] = cells.cluster.map(grole)
_g = [cl for cl in g_clusters if grole.get(cl, "").startswith("mitotic")]
_n = [cl for cl in g_clusters if grole.get(cl, "").startswith("endocycling")]
if _g and _n and ze.notna().any() and ze[_g].mean() < ze[_n].mean():
    print("NOTE: bam is not higher in the group called germarium. bam is on in only a handful of cells, so this "
          "often happens and is not by itself a problem; but if the M-phase separation is also weak, set "
          "GERMARIUM_CLUSTERS by hand from the table above.")
if _g and _n and n_counts[_n].mean() < n_counts[_g].mean():
    print("NOTE: the group called nurse cells does not have more counts per cell than the germarium group. "
          "Nurse cells are far larger, so check the split before trusting the result.")
GMIT, GENDO = "mitotic germline (germarium)", "endocycling germline (nurse cells)"
n_ok = [r for r in (GMIT, GENDO) if (cells.gl_role == r).sum() >= 30]
if len(n_ok) < 2:
    raise StopHere("Could not separate dividing germarial germline from endocycling nurse cells "
                   f"(found {cells.gl_role.value_counts().to_dict()}). Raising N_CLUSTERS in Cell 1 often "
                   "splits them; if the dataset has no germarium, this comparison cannot be made here.")

# Same stress matching as the follicle comparison
cut_g = min(np.quantile(cells.stress[cells.gl_role == GMIT], 0.75),
            np.quantile(cells.stress[cells.gl_role == GENDO], 0.75))
low_g = cells.stress <= cut_g
pb, lab = [], []
for lib in sorted(cells.library.unique()):
    for r in (GMIT, GENDO):
        m = ((cells.library == lib) & (cells.gl_role == r) & low_g).to_numpy()
        if m.sum() >= 30:
            pb.append(np.asarray(X[np.flatnonzero(m)].sum(axis=0)).ravel()); lab.append((lib, r))
if len({l for l, r in lab if (l, GMIT) in lab and (l, GENDO) in lab}) == 0:
    raise StopHere("No library has at least 30 cells of both germline groups after stress matching.")
pbg = pd.DataFrame(np.array(pb).T, index=genes, columns=pd.MultiIndex.from_tuples(lab))
pbg = pbg[(pbg / pbg.sum() * 1e6 >= 5).all(axis=1)]
lg = np.log(pbg.replace(0, np.nan)); refg = lg.mean(axis=1)
sizeg = np.exp(lg[refg.notna()].sub(refg[refg.notna()], axis=0).median())
Lg = np.log2(pbg / sizeg / pbg.sum().mean() * 1e6 + 1)
fc_gl = Lg.xs(GENDO, level=1, axis=1).mean(axis=1) - Lg.xs(GMIT, level=1, axis=1).mean(axis=1)
print(f"\nGermline pseudobulk: {len(lab)} groups, {len(Lg):,} genes expressed in all of them")

# Positive control: nurse cells must look less mitotic and more endoreplicated than germarial cysts
# The control has to be strict. Germarial cysts divide; nurse cells never do, so M-phase genes should fall
# steeply between them. A shallow fall means both groups are endocycling - that is, early against late nurse
# cells - and that comparison is NOT the mitotic-to-endocycle switch this cell exists to test.
mitg = [g for g in MPHASE if g in fc_gl.index]
drop = float(fc_gl.loc[mitg].median()) if mitg else float("nan")
print(f"\nPositive control, M-phase genes in nurse cells vs germarium: {drop:+.2f} ({len(mitg)} genes)")
_bam = resolve("bam", set(genes))
if _bam and _bam in gpos:
    bi = gpos[_bam]
    for r in (GMIT, GENDO):
        col = X[np.flatnonzero((cells.gl_role == r).to_numpy())][:, bi]
        print(f"   cells expressing {_bam}: {r} {100 * float((col.toarray().ravel() > 0).mean()):.1f}%")
GERMLINE_USABLE = bool(mitg) and drop <= -0.8
if not GERMLINE_USABLE:
    print("""
WARNING - the germline comparison did not pass its control, so DO NOT read the verdicts below as
follicle-versus-germline. M-phase genes barely fall, which means both germline groups are almost certainly
endocycling already: this is early against late nurse cells, not germarium against nurse cells. A true
germarium is a small structure and many ovary libraries barely sample it.
What to try, in order:
  1. Raise N_CLUSTERS in Cell 1 and run again - a germarial cluster is often hidden inside a larger one.
  2. Run the other atlas. GSE136162 has three libraries and samples the germarium better.
  3. If neither separates a dividing germline population, this dataset cannot answer the question and the
     follicle-cell result has to stand on its own.
""")

levg = Lg.mean(axis=1)
binsg = pd.qcut(levg.rank(method="first"), 10, labels=False)
avoidg = {g for v in PROGRAMMES.values() for g in v}
poolsg = {b: [g for g in binsg.index[binsg == b] if g not in avoidg] for b in range(10)}
rows = []
for n, gs in PROGRAMMES.items():
    gs = [g for g in gs if g in fc_gl.index]
    if len(gs) < 3:
        continue
    obs = float(fc_gl.loc[gs].median())
    null = np.array([fc_gl.loc[[rng.choice(poolsg[binsg[g]]) for g in gs]].median() for _ in range(N_RANDOM_SETS)])
    c = float(np.median(null))
    rows.append({"programme": n, "genes": len(gs), "germline": obs, "random_median": c,
                 "p_vs_random": (1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (N_RANDOM_SETS + 1),
                 "pct_of_all_genes": float(fc_gl.rank(pct=True).loc[gs].median()) * 100})
resg = pd.DataFrame(rows)
foll = res_follicle.set_index("programme")["log2_change"]
resg["follicle"] = resg.programme.map(foll)
resg["both_same_direction"] = np.sign(resg.germline) == np.sign(resg.follicle)
print("\nProgrammes, germline against follicle cells:")
display(resg[["programme", "genes", "follicle", "germline", "pct_of_all_genes",
              "p_vs_random", "both_same_direction"]].round(3))

KEY = ["Jafrac1", "Jafrac2", "Ero1L", "Gclc", "Trx-2", "Trxr-1", "Rrp1", "Hsc70-3", "Sec61beta", "Cat", "geminin"]
key = [resolve(g, set(fc_gl.index)) for g in KEY]
key = [(k, g) for k, g in zip(KEY, key) if g and g in fc_follicle.index]
tab_k = pd.DataFrame({"follicle": [float(fc_follicle.loc[g]) for _, g in key],
                      "germline": [float(fc_gl.loc[g]) for _, g in key],
                      "germline_pct": [float(fc_gl.rank(pct=True).loc[g]) * 100 for _, g in key]},
                     index=[k for k, _ in key])
fpct = fc_follicle.rank(pct=True)
tab_k["follicle_pct"] = [float(fpct.loc[g]) * 100 for _, g in key]


def verdict(fp, gp):
    """Judge each lineage against its own genome rather than against zero."""
    w = lambda p: "up" if p > 65 else "down" if p < 35 else "flat"
    a, b = w(fp), w(gp)
    if a == "flat" and b == "flat":
        return "neither"
    if a == b:
        return "BOTH lineages - tracks endocycling"
    if b == "flat":
        return "follicle only - likely differentiation"
    if a == "flat":
        return "germline only"
    return "opposite directions"


tab_k["verdict"] = [verdict(f, g) for f, g in zip(tab_k.follicle_pct, tab_k.germline_pct)]
tab_k = tab_k[["follicle", "follicle_pct", "germline", "germline_pct", "verdict"]]
hdr = ("\nGene by gene. Each lineage is judged against its own genome, so a value near zero reads as flat "
       "rather than as agreement:")
if not GERMLINE_USABLE:
    hdr += ("\n*** The control above did not pass, so the 'germline' column is early vs late nurse cells, not "
            "the switch. Every verdict in this table is unreliable until that is fixed. ***")
print(hdr)
display(tab_k.round(2))
print("""
How to read this. A change that appears in BOTH lineages tracks the mitotic-to-endocycle switch itself, because
the only thing nurse cells and follicle cells share here is that switch. A change in follicle cells ALONE is
most likely secretory differentiation, which is what the salivary gland comparison could never exclude.
Two cautions: nurse cells are germline, so germline-restricted genes such as dhd are not comparable across the
two lineages; and they are far larger than follicle cells, which is why the pseudobulk uses median-of-ratios.
""")
fc_gl.to_csv(os.path.join(GDIR, "germline_fold_changes.csv"))
resg.to_csv(os.path.join(GDIR, "germline_programmes.csv"), index=False)
tab_k.to_csv(os.path.join(GDIR, "germline_vs_follicle_genes.csv"))
print("Saved germline tables to", GDIR)

In [ ]:
#@title Cell 11 - Unbiased: what actually changes across the switch, every gene
# Every cell so far asked about genes chosen in advance. This one asks the opposite question: take the whole
# transcriptome, rank it by how much it changes from mitotic to endocycling, and look at what comes out. That
# is the only way to find out what the switch is mostly doing, rather than what we went looking for.
#
# Three things are reported for every gene, and the third is the one that matters:
#   1. log2 fold change, endocycling vs mitotic follicle cells (pseudobulk, median-of-ratios).
#   2. Whether every library agrees on the direction. With 3 libraries this is real replication; with 1 it is
#      not, and the column says so.
#   3. The same change in the GERMLINE. Shared with the germline -> the endocycle. Follicle only -> secretory
#      differentiation. This is what stopped the redox leads being believed, and it applies to every gene here.
#
# No p-values at gene level. The groups are cells, not animals, so per-cell tests would report thousands of
# significant genes off a handful of libraries. Effect size plus direction agreement across libraries plus the
# germline comparison is the honest version of the same question.

MIN_DETECT = 0.10          #@param {type:"number"}
TOP_N = 40                 #@param {type:"integer"}

_mitx = np.flatnonzero(((cells.role == MIT) & cells.low_stress).to_numpy())
_endx = np.flatnonzero(((cells.role == ENDO) & cells.low_stress).to_numpy())
order = [g for g in fc_follicle.index if g in gpos]
cols = [gpos[g] for g in order]
det_m = pd.Series(np.asarray((X[_mitx][:, cols] > 0).mean(axis=0)).ravel(), index=order)
det_e = pd.Series(np.asarray((X[_endx][:, cols] > 0).mean(axis=0)).ravel(), index=order)

de = pd.DataFrame({"log2FC": fc_follicle.reindex(order),
                   "detect_mitotic": det_m, "detect_endo": det_e})
de["expr_level"] = [float(Lp.loc[g].mean()) for g in order]
de["expr_rank_pct"] = de.expr_level.rank(pct=True) * 100

libcols2 = sorted(per_lib)
if len(libcols2) > 1:
    sgn = pd.DataFrame({f"lib_{str(l)[:14]}": per_lib[l].reindex(order) for l in libcols2})
    de["libraries_agree"] = (np.sign(sgn).nunique(axis=1) == 1)
    de = pd.concat([de, sgn.round(2)], axis=1)
else:
    de["libraries_agree"] = np.nan

# The germline column, and the verdict that goes with it.
if "fc_gl" in dir():
    gl_pct = fc_gl.rank(pct=True) * 100
    f_pct = fc_follicle.rank(pct=True) * 100
    de["germline_log2FC"] = fc_gl.reindex(order)
    de["germline_pct"] = gl_pct.reindex(order)
    de["follicle_pct"] = f_pct.reindex(order)
    # Cell 10's verdict judges each lineage against its own genome, which is right for a handful of chosen
    # genes but far too permissive across 10,000: a gene moving 4.4 in follicle cells and 0.3 in the germline
    # would read as "both". Across the whole transcriptome the germline change has to be a real change too,
    # so a minimum effect size is required on that side as well.
    MIN_GL_EFFECT = 0.3

    def lineage_call(f_lfc, g_lfc, fp, gp):
        if pd.isna(g_lfc) or pd.isna(f_lfc):
            return "not in germline data"
        f_moves = abs(f_lfc) >= MIN_GL_EFFECT and (fp > 65 or fp < 35)
        g_moves = abs(g_lfc) >= MIN_GL_EFFECT and (gp > 65 or gp < 35)
        if not f_moves and not g_moves:
            return "neither"
        if f_moves and g_moves:
            return ("BOTH lineages - tracks endocycling" if np.sign(f_lfc) == np.sign(g_lfc)
                    else "opposite directions")
        return "follicle only - likely differentiation" if f_moves else "germline only"

    de["lineage"] = [lineage_call(f, g, fp, gp) for f, g, fp, gp in
                     zip(de.log2FC, de.germline_log2FC, de.follicle_pct, de.germline_pct)]
else:
    de["lineage"] = "Cell 10 not run"

# Annotate anything that was in a curated set, so a familiar gene is recognisable in the ranked list.
member = {}
for nm, gs in (PROGRAMMES.items() if "PROGRAMMES" in dir() else []):
    for g in gs:
        member.setdefault(g, []).append(nm)
de["in_set"] = [", ".join(member.get(g, [])) for g in de.index]

keep = de[(de[["detect_mitotic", "detect_endo"]].max(axis=1) >= MIN_DETECT)].copy()
print(f"{len(de):,} genes in the comparison; {len(keep):,} detected in at least {MIN_DETECT:.0%} of cells in "
      f"one group or the other. The rest are too sparse to rank.")
if len(libcols2) > 1:
    agree_only = keep[keep.libraries_agree]
    print(f"Of those, {len(agree_only):,} change in the same direction in all {len(libcols2)} libraries. "
          f"Rankings below use that subset.")
    rank_on = agree_only
else:
    print("Only one library has both cell groups, so there is no direction check and nothing below is "
          "replicated. Treat the ranking as a list of candidates, not results.")
    rank_on = keep

show = ["log2FC", "detect_mitotic", "detect_endo", "expr_rank_pct", "germline_log2FC", "lineage", "in_set"]
show = [c for c in show if c in rank_on.columns]
print(f"\n=== TOP {TOP_N} UP in endocycling follicle cells ===")
display(rank_on.nlargest(TOP_N, "log2FC")[show].round(2))
print(f"\n=== TOP {TOP_N} DOWN in endocycling follicle cells ===")
display(rank_on.nsmallest(TOP_N, "log2FC")[show].round(2))

# The subset that matters most: large changes that the germline makes too.
if "germline_log2FC" in rank_on.columns:
    both = rank_on[rank_on.lineage.astype(str).str.startswith("BOTH")]
    both = both.reindex(both.log2FC.abs().sort_values(ascending=False).index)
    print(f"\n=== Changes shared with the GERMLINE: {len(both):,} genes ===")
    print("These are the candidates for tracking the endocycle itself rather than secretory differentiation.")
    display(both.head(TOP_N)[show].round(2))
    n_fol = int((rank_on.lineage.astype(str).str.startswith("follicle only")).sum())
    print(f"\nFor scale: {len(both):,} genes shared with the germline against {n_fol:,} follicle-only. "
          "A gene appearing here is necessary but not sufficient - the germline split is a two-group "
          "comparison with its own noise, so treat this as the shortlist to check, not a result.")

out = os.path.join(GDIR, "route_e_unbiased_de.csv")
de.sort_values("log2FC").to_csv(out)
print("\nSaved the full ranked table (every gene, unfiltered) to", out)
print("Columns: log2FC, detection in each group, expression rank, per-library fold changes, the germline "
      "value and the lineage verdict. Open it in a spreadsheet and sort on whatever you like.")

In [ ]:
#@title Cell 12 - Is the germline signal real, or follicle mRNA leaking into germline droplets?
# Cell 11 called 700-odd genes "shared with the germline". The list cannot be taken at face value, and the
# reason is visible in it: the top of the shared list is Yp1, Yp2, Yp3, Vm34Ca, Vm26Aa, Vm26Ab. Vitelline
# membrane genes are made by FOLLICLE cells, not by nurse cells. They have no business changing in the
# germline at all, so their presence says the germline column is partly picking up follicle transcripts.
#
# How that happens: free mRNA from lysed cells is captured along with every cell, so a transcript that a
# neighbouring follicle cell is producing in enormous quantity appears at a low level in germline droplets
# too. Nurse cells sit inside egg chambers whose follicle cells are pouring out eggshell and yolk message,
# while germarial cysts do not - so ambient mRNA alone produces a germline "change" in the same direction as
# the follicle one. That is exactly the artefact that would manufacture shared calls.
#
# The check: a real germline change should be in a gene the germline actually expresses. Two requirements,
# and the Vm genes are the built-in test of whether the check works - if they are not flagged, it is not.
MIN_GL_DETECT = 0.25       #@param {type:"number"}
MAX_LEVEL_GAP = 2.0        #@param {type:"number"}

_gm = np.flatnonzero(((cells.gl_role == GMIT) & low_g).to_numpy())
_ge = np.flatnonzero(((cells.gl_role == GENDO) & low_g).to_numpy())
gl_order = [g for g in Lg.index if g in gpos]
gl_cols = [gpos[g] for g in gl_order]
gdet_m = pd.Series(np.asarray((X[_gm][:, gl_cols] > 0).mean(axis=0)).ravel(), index=gl_order)
gdet_e = pd.Series(np.asarray((X[_ge][:, gl_cols] > 0).mean(axis=0)).ravel(), index=gl_order)

gl_level = Lg.mean(axis=1)
fol_level = Lp.mean(axis=1)
gap = (fol_level - gl_level).reindex(gl_order)   # positive = much higher in follicle cells than germline

chk = pd.DataFrame({"gl_detect_germarium": gdet_m, "gl_detect_nurse": gdet_e,
                    "gl_level": gl_level.reindex(gl_order), "follicle_level": fol_level.reindex(gl_order),
                    "level_gap": gap})
chk["gl_detect_max"] = chk[["gl_detect_germarium", "gl_detect_nurse"]].max(axis=1)
chk["bleed_through_risk"] = np.where(
    chk.gl_detect_max < MIN_GL_DETECT, "germline barely expresses it",
    np.where(chk.level_gap > MAX_LEVEL_GAP, f"{2 ** MAX_LEVEL_GAP:.0f}x lower in germline than follicle", "ok"))

de2 = de.join(chk[["gl_detect_max", "level_gap", "bleed_through_risk"]])
de2["shared_confirmed"] = (de2.lineage.astype(str).str.startswith("BOTH")) & (de2.bleed_through_risk == "ok")

# Does the check catch the genes we already know are follicle-made? If not, stop reading here.
KNOWN_FOLLICLE = [g for g in "Vm34Ca Vm26Aa Vm26Ab Vm32E Cp36 Cp38 Cp15 Cp16 dec-1 psd".split() if g in de2.index]
if KNOWN_FOLLICLE:
    print("Validation of the check itself. These are eggshell genes, transcribed by follicle cells and not by "
          "nurse cells, so every one of them should be flagged:")
    display(de2.loc[KNOWN_FOLLICLE, ["log2FC", "germline_log2FC", "gl_detect_max", "level_gap",
                                     "bleed_through_risk"]].round(2))
    caught = (de2.loc[KNOWN_FOLLICLE, "bleed_through_risk"] != "ok").mean()
    print(f"Flagged: {caught:.0%} of them.")
    if caught < 0.6:
        print("*** The check is NOT catching genes known to be follicle-made. Do not use the verdicts below; "
              "lower MIN_GL_DETECT or raise MAX_LEVEL_GAP and look at the numbers for these genes by hand. ***")

shared0 = int(de2.lineage.astype(str).str.startswith("BOTH").sum())
shared1 = int(de2.shared_confirmed.sum())
print(f"\nShared with the germline before this check: {shared0:,} genes. After it: {shared1:,}.")
print(f"Removed: {shared0 - shared1:,} genes the germline barely expresses, or expresses far below the "
      f"follicle level, where ambient mRNA is the simpler explanation.")

keep2 = de2[de2.shared_confirmed & (de2.libraries_agree.fillna(True))]
keep2 = keep2.reindex(keep2.log2FC.abs().sort_values(ascending=False).index)
cols2 = [c for c in ["log2FC", "germline_log2FC", "detect_endo", "gl_detect_max", "level_gap", "expr_rank_pct",
                     "in_set"] if c in keep2.columns]
print(f"\n=== Shared with the germline AND expressed in the germline: {len(keep2):,} genes ===")
print("This is the shortlist for changes that track the endocycle rather than secretory differentiation.")
display(keep2.head(50)[cols2].round(2))

# Where the curated redox genes land once the check is applied.
WATCH = [g for g in "Jafrac1 Jafrac2 Ero1L Gclc Trx-2 Trxr-1 Rrp1 Ogg1 mre11 Cat Hsc70-3 Sec61beta geminin".split()
         if g in de2.index]
print("\nThe genes this project has been following, with the same check applied:")
display(de2.loc[WATCH, [c for c in ["log2FC", "germline_log2FC", "gl_detect_max", "level_gap",
                                    "bleed_through_risk", "lineage"] if c in de2.columns]].round(2))
print("""
Read the direction of the bias before reading anything else. Ambient follicle mRNA pushes the germline column
TOWARDS the follicle one, so it manufactures agreement and cannot manufacture disagreement. A redox gene that
still reads flat in the germline is therefore a stronger negative after this check than before it, not a weaker
one. What the check can overturn is the shared list, not the follicle-only calls.
""")
de2.sort_values("log2FC").to_csv(os.path.join(GDIR, "route_e_unbiased_de_checked.csv"))
print("Saved to", os.path.join(GDIR, "route_e_unbiased_de_checked.csv"))